In [ ]:
from config_dirs import OPENAI_INPUT, OPENAI_ANALYSIS_RESULTS_DIR, OPENAI_FULL_OUTPUT
import os, pandas as pd
import openai
import time
import logging

logging.basicConfig(level=logging.INFO)

# Create output folder if it doesn't exist
os.makedirs(OPENAI_ANALYSIS_RESULTS_DIR, exist_ok=True)
# -------------------------------
# Configuration
# -------------------------------
openai.api_key = 'your_api_key_here'
input_file = OPENAI_INPUT  
batch_size = 100                   # N of rows per batch
os.makedirs(OPENAI_ANALYSIS_RESULTS_DIR, exist_ok=True)

# -------------------------------
# Function to analyze abstracts
# -------------------------------
def analyze_abstract(abstract):
    max_retries = 10
    retry_delay = 10  # in seconds
    for attempt in range(max_retries):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-4o-2024-11-20",
                messages=[
                    {
                        "role": "system",
                        "content": """
                        You are an AI assistant specializing in genomics and vascular diseases, with expertise in genomic and medical terminology, statistical analysis, and understanding of causality in medicine. Your task is to analyze PubMed abstracts to identify genes or variables associated with vascular diseases. You will also determine whether these associations are causal or consequential. Follow these steps for each abstract:
                        1. 1. Identify any mention of genes, mutations, or other relevant variables (e.g., proteins, biomarkers) associated with vascular diseases.
                        2. Identify the entity (Gene, Protein, Biomarker, or other variable).
                        3. Specify the name of the entity (e.g., ADAMTS13, MRAS, lectin-like oxLDL receptor 1, HDL-C).
                        4. Determine the type of variability (e.g., Missense mutation, Nonsense mutation, Frameshift mutation, Single-nucleotide polymorphism [SNP], Upregulation, Downregulation, etc.).
                        5. Indicate if the association is causal or consequential, taking into account the study design, statistical evidence, and biological plausibility. Where possible, mention the type of study (GWAS, observational, functional assay) that supports your conclusion. Note that abstract-only analysis may limit the depth of this determination.
                        6. Include statistical measures of significance if available (e.g., effect sizes, p-values, odds ratios). If no significance is provided, mention the methods used to obtain the data (e.g., “detected by Genome Wide Association Studies”, “detected by Whole Exome Sequencing” ). If more than one method is mentioned, list all relevant major methods.
                        7. Identify Diseases/Pathologies associated with every identified entity/variable.
                        8. Identify the organism in the study (e.g., human, mouse).
                        9. Identify the organs or systems under study (e.g., kidney, brain, heart, liver, cardiovascular system, cerebrovascular system).
                        10. Identify the sample size mentioned in the abstract related to the study population or cohort, and link it to the relevant findings. If not specified, write “Not specified.”
                        11. Identify the population ethnicity mentioned in the abstract and link it to the relevant findings. If not specified, write “Not specified.”
                        12.Provide the exact sentence (or a representative sentence if there are multiple) from the abstract supporting the association.
                        13. Generate a structured output in a tab-delimited format with the following columns: Entity of Association⇥Name⇥Variability Type⇥Causality⇥Significance⇥Disease/Pathology⇥Organism⇥Organ/System⇥Sample Size⇥Population Ethnicity⇥Abstract Reference.
                        If no relevant gene/variable associations are found, respond with 'result: not_found.'.
                        Here are some examples to guide your output:
                        Example 1:
                        Abstract: "We describe two brothers from a consanguineous family of Egyptian ancestry, presenting with microcephaly, apparent global developmental delay, seizures, spasticity, congenital blindness, and multiple cutaneous capillary malformations. Through exome sequencing, we uncovered a homozygous missense variant in STAMBP (p.K303R) in the two siblings, inherited from heterozygous carrier parents. Mutations in STAMBP are known to cause microcephaly-capillary malformation syndrome (MIC-CAP) and the phenotype in this family is consistent with this diagnosis. We compared the findings in the present brothers with those of earlier reported patients. © 2016 Wiley Periodicals, Inc."
                        Entity of Association⇥Name⇥Variability Type⇥Causality⇥Significance⇥Disease/Pathology⇥Organism⇥Organ/System⇥Sample Size⇥Population Ethnicity⇥Abstract Reference
                        Gene⇥STAMBP⇥Missense variant (p.K303R)⇥Causal⇥Detected by exome sequencing⇥Microcephaly-capillary malformation syndrome (MIC-CAP)⇥Human⇥Central Nervous System(Brain), Visual system(Eye), Vascular System, Skin⇥N=2 (Two brothers)⇥Egyptian⇥"Through exome sequencing, we uncovered a homozygous missense variant in STAMBP (p.K303R) in the two siblings, inherited from heterozygous carrier parents. Mutations in STAMBP are known to cause microcephaly-capillary malformation syndrome (MIC-CAP) and the phenotype in this family is consistent with this diagnosis."
                        Example 2:
                        Abstract: "To investigate the prevalence of the G20210A prothrombin and G1691A factor V gene variants in patients with acute coronary syndrome stratified according to risk factor profile and to extent of coronary disease, in comparison with matched healthy controls. The 20210 prothrombin and the 1691 factor V loci were genotyped in 247 patients < or =65 years of age (190 myocardial infarction and 57 unstable angina as first presentation of disease) and in 247 healthy age- and sex-matched controls. The prevalence of the 1691A factor V allele was similar in cases and controls. The frequency of heterozygotes for the 20210A prothrombin allele was 6.5% among patients and 2.8% among controls (OR 2.4, 95% CI 1.0-5.9), increasing to 8.7% in patients with a family history of myocardial infarction (OR 3.3, 95% CI 1.2-9.1), to 9.9% in patients (n=81) with < or =1 vessel disease (OR 3.8, 95% CI 1.3-10.8), and to 13.0% in patients who were normocholesterolaemic, non-diabetic, normotensive and non-smokers (OR 5.1, 95% CI 1.2-21.4). These findings suggest that the 20210A prothrombin allele represents an inherited risk factor for acute coronary syndrome among patients who have limited extent of coronary disease at angiography or who lack major metabolic and acquired risk factors."
                        Entity of Association⇥Name⇥Variability Type⇥Causality⇥Significance⇥Disease/Pathology⇥Organism⇥Organ/System⇥Sample Size⇥Population Ethnicity⇥Abstract Reference
                        Gene⇥Prothrombin (F2) G20210A⇥Heterozygous Point Mutation (SNV) ⇥Causal (risk factor)⇥OR=2.4 (95% CI 1.0–5.9); OR=3.3 (95% CI 1.2–9.1); OR=3.8 (95% CI 1.3–10.8); OR=5.1 (95% CI 1.2–21.4)⇥Acute coronary syndrome (Myocardial Infarction, unstable angina)⇥Human⇥Cardiovascular System⇥N=494 (247 patients and 247 controls)⇥Not Specified⇥"The frequency of heterozygotes for the 20210A prothrombin allele was 6.5% among patients and 2.8% among controls (OR 2.4, 95% CI 1.0-5.9), increasing to 8.7% in patients with a family history of myocardial infarction (OR 3.3, 95% CI 1.2-9.1), to 9.9% in patients (n=81) with < or =1 vessel disease (OR 3.8, 95% CI 1.3-10.8), and to 13.0% in patients who were normocholesterolaemic, non-diabetic, normotensive and non-smokers (OR 5.1, 95% CI 1.2-21.4). These findings suggest that the 20210A prothrombin allele represents an inherited risk factor for acute coronary syndrome."                                        
                        Example 3:
                        Abstract: "Mutations in the MYH9 gene, which encodes the nonmuscle myosin heavy chain IIA, have been recently reported in three syndromes that share the association of macrothrombocytopenia (MTCP) and leukocyte inclusions: the May-Hegglin anomaly and Sebastian and Fechtner syndromes. Epstein syndrome, which associates inherited sensorineural deafness, glomerular nephritis, and MTCP without leukocyte inclusions, was shown to be genetically linked to the same locus at 22q12.3 to 13. The expression of MYH9 in the fetal and mature human kidney was studied, and the 40 coding exons of the gene were screened by single-strand conformation polymorphism in 12 families presenting with the association of MTCP and nephropathy. MYH9 is expressed in both fetal and mature kidney. During renal development, it is expressed in the late S-shaped body, mostly in its lower part, in the endothelial and the epithelial cell layers. Later, as well as in mature renal tissue, MYH9 is widely expressed in the kidney, mainly in the glomerulus and peritubular vessels. Within the glomerulus, MYH9 mRNA and protein are mostly expressed in the epithelial visceral cells. Four missense heterozygous mutations that are thought to be pathogenic were found in five families, including two families with Epstein syndrome. Three mutations were located in the coiled-coil rod domain of the protein, and one was in the motor domain. Two mutations (E1841K and D1424N) have been reported elsewhere in families with May-Hegglin anomaly. The two others (R1165L and S96L) are new mutations, although one of them affects a codon (R1165), found elsewhere to be mutated in Sebastian syndrome."
                        Entity of Association⇥Name⇥Variability Type⇥Causality⇥Significance⇥Disease/Pathology⇥Organism⇥Organ/System⇥Sample Size⇥Population Ethnicity⇥Abstract Reference
                        Gene⇥MYH9⇥Heterozygous Missense mutations (E1841K, D1424N, R1165L, S96L)⇥Causal (likely pathogenic)⇥Reports from "expression studies" and "screened by single-strand conformation polymorphism"⇥Macrothrombocytopenia, May-Hegglin anomaly, Sebastian syndrome, Fechtner syndrome, Epstein syndrome	⇥Human⇥Kidney (glomeruli, peritubular vessels), Hematologic system (platelets), Auditory system (inner ear)⇥N=12 families (including 5 with identified mutations)⇥Not Specified⇥"Four missense heterozygous mutations that are thought to be pathogenic were found in five families, including two families with Epstein syndrome."  
                        Example 4:
                        Abstract: "Cardiovascular disease (CVD) affecting blood vessel function is a leading cause of death around the world. A common treatment option to replace the diseased blood vessels is vascular grafting using the patient's own blood vessels. However, patients with CVD are usually lacking vessels for grafting. Recent advances in tissue engineering are now providing alternatives to autologous vascular grafts in the form of tissue-engineered blood vessels (TEBVs). In this review, we will describe the use of different scaffolding systems, cell sources and conditioning approaches for creating fully functional blood vessels. Additionally, we will present the methods used for assessing TEBV functions and describe preclinical and clinical trials for TEBV. Although the early results were encouraging, current designs of TEBV still fall short as a viable clinical option. Implementing the current knowledge in vascular development can lead to improved fabrication and function of TEBV and hasten clinical translation."
                        result: not_found.
                        Example 5:
                        Abstract: "Genome-wide association studies have revealed an association between coronary heart disease (CHD) and genetic variation on chromosome 13q34, with the lead single nucleotide polymorphism rs4773144 residing in the COL4A2 gene in this genomic region. We investigated the functional effects of this genetic variant. Analyses of primary cultures of vascular smooth muscle cells (SMCs) and endothelial cells (ECs) from different individuals showed a difference between rs4773144 genotypes in COL4A2 and COL4A1 expression levels, being lowest in the G/G genotype, intermediate in A/G and highest in A/A. Chromatin immunoprecipitation followed by allelic imbalance assays of primary cultures of SMCs and ECs that were of the A/G genotype revealed that the G allele had lower transcriptional activity than the A allele. Electrophoretic mobility shift assays and luciferase reporter gene assays showed that a short DNA sequence encompassing the rs4773144 site interacted with a nuclear protein, with lower efficiency for the G allele, and that the G allele sequence had lower activity in driving reporter gene expression. Analyses of cultured SMCs from different individuals demonstrated that cells of the G/G genotype had higher apoptosis rates. Immunohistochemical and histological examinations of ex vivo atherosclerotic coronary arteries from different individuals disclosed that atherosclerotic plaques with the G/G genotype had lower collagen IV abundance and thinner fibrous cap, a hallmark of unstable, rupture-prone plaques. A study of a cohort of patients with angiographically documented coronary artery disease showed that patients of the G/G genotype had higher rates of myocardial infarction, a phenotype often caused by plaque rupture. These results indicate that the CHD-related genetic variant at the COL4A2 locus affects COL4A2/COL4A1 expression, SMC survival, and atherosclerotic plaque stability, providing a mechanistic explanation for the association between the genetic variant and CHD risk."
                        Entity of Association⇥Name⇥Variability Type⇥Causality⇥Significance⇥Disease/Pathology⇥Organism⇥Organ/System⇥Sample Size⇥Population Ethnicity⇥Abstract Reference
                        Gene⇥COL4A2 (rs4773144)⇥Single-nucleotide polymorphism (SNP)⇥Causal (risk factor)⇥Identified by GWAS; mechanistic assays (chromatin IP, luciferase tests); no explicit p-values⇥Coronary heart disease (CHD), Myocardial infarction (MI), Atherosclerotic plaque instability⇥Human⇥Cardiovascular system⇥Not specified⇥Not specified⇥"These results indicate that the CHD-related genetic variant at the COL4A2 locus affects COL4A2/COL4A1 expression, SMC survival, and atherosclerotic plaque stability, providing a mechanistic explanation for the association between the genetic variant and CHD risk."
                        Gene⇥COL4A1⇥Expression changes (downregulated in G allele; upregulated in A allele)⇥Consequential⇥Functional assays (expression analyses); no explicit p-values⇥Coronary heart disease (CHD), Myocardial infarction (MI), Atherosclerotic plaque instability⇥Human⇥Cardiovascular system⇥Not specified⇥Not specified⇥"Analyses of primary cultures of vascular smooth muscle cells (SMCs) and endothelial cells (ECs) from different individuals showed a difference between rs4773144 genotypes in COL4A2 and COL4A1 expression levels..."
                        Protein⇥Collagen IV⇥Reduced abundance in G/G genotype⇥Consequential⇥Immunohistochemical analysis; no explicit p-values⇥Coronary heart disease (CHD), Myocardial infarction (MI), Atherosclerotic plaque instability⇥Human⇥Cardiovascular system⇥Not specified⇥Not specified⇥"Atherosclerotic plaques with the G/G genotype had lower collagen IV abundance and thinner fibrous cap, a hallmark of unstable, rupture-prone plaques."
                        """
                    },
                    {
                        "role": "user",
                        "content": abstract
                    }
                ],
                temperature=0,
                max_tokens=2300
            )
            return response['choices'][0]['message']['content']
        except openai.error.RateLimitError as e:
            print(f"Rate limit error: {e}. Retrying in {retry_delay} seconds...")
            time.sleep(retry_delay)
            retry_delay *= 2
        except Exception as e:
            print(f"An error occurred: {e}")
            break
    return 'result: not_found'

# -------------------------------
# Function to analyze a batch and save results
# -------------------------------
def analyze_and_save(df_batch):
    for _, row in df_batch.iterrows():
        pmid = row['PMID']
        abstract = row['Abstract']

        result = analyze_abstract(abstract)
        print(f"Analysis result for PMID {pmid}:\n{result}\n")

        output_file = os.path.join(output_dir, f'{pmid}.txt')
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(result)

# -------------------------------
# Read input file and divide into batches
# -------------------------------
df = pd.read_excel(input_file)
num_batches = (len(df) + batch_size - 1) // batch_size

for batch_idx in range(num_batches):
    start = batch_idx * batch_size
    end = start + batch_size
    batch_df = df.iloc[start:end]
    logging.info(f"Processing batch {batch_idx + 1}/{num_batches} ({len(batch_df)} rows)")
    analyze_and_save(batch_df)

# -------------------------------
# Collect "not_found" results
# -------------------------------
not_found_results = []
for filename in os.listdir(output_dir):
    if filename.endswith('.txt'):
        with open(os.path.join(output_dir, filename), 'r', encoding='utf-8') as f:
            content = f.read()
            if "result: not_found" in content:
                not_found_results.append(f"Analysis result for PMID {filename.replace('.txt','')}:\n{content}")

# Save "not_found" results to a separate file
not_found_file = 'not_found_collection.txt'
with open(not_found_file, 'w', encoding='utf-8') as f:
    for entry in not_found_results:
        f.write(entry + "\n")
print(f"Saved not_found results to {not_found_file}")

# -------------------------------
# Remove files with "not_found" results
# -------------------------------
for entry in not_found_results:
    pmid = entry.split('\n')[0].split(' ')[-1].strip(':')
    file_to_delete = os.path.join(output_dir, f'{pmid}.txt')
    if os.path.exists(file_to_delete):
        os.remove(file_to_delete)
        print(f"Deleted file: {file_to_delete}")

# -------------------------------
# Combine valid results into a single CSV
# -------------------------------
df_list = []
for filename in os.listdir(output_dir):
    if filename.endswith('.txt'):
        file_path = os.path.join(output_dir, filename)
        try:
            temp_df = pd.read_csv(file_path, sep='⇥', engine='python', quoting=3)
            temp_df['PMID'] = filename.replace('.txt','')
            df_list.append(temp_df)
        except Exception as e:
            print(f"Error processing file {filename}: {e}")

if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    cols = ['PMID'] + [col for col in combined_df.columns if col != 'PMID']
    combined_df = combined_df[cols]
    combined_df.to_csv(OPENAI_FULL_OUTPUT, index=False)
    print("All valid results combined and saved to OpenAI_full_output.csv")
else:
    print("No valid data to combine.")

print("Analysis complete.")
